# Q8 — 멀티태스크(결함 + 공정 원인) 장기학습 + TTA
Q5의 멀티태스크 구조에 Q6 레시피(50에폭 + 조기종료 + TTA)를 적용해 두 헤드를 모두 끌어올린다.
원인 라벨은 결함→원인 1:1 매핑에서 생성하므로 원인 F1은 결함 F1을 따라간다. 목표: 결함·원인 macro-F1 둘 다 0.91대.
비교 기준: Q5(결함 0.890 / 원인 0.896).

In [ ]:
# --- 한글 폰트 + import ---
import sys
from pathlib import Path
_p=Path.cwd()
for c in [_p,_p.parent,_p.parent.parent]:
    if (c/"utils").is_dir(): sys.path.insert(0,str(c)); PROJ=c; break
else: PROJ=_p
from utils.korean_font import set_korean_font
set_korean_font()
import numpy as np, matplotlib.pyplot as plt, time, itertools, yaml
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

CLASSES=["Center","Donut","Edge-Loc","Edge-Ring","Loc","Near-full","Random","Scratch","none"]
NUM=len(CLASSES)
DEVICE="cuda" if torch.cuda.is_available() else "cpu"
PROC=PROJ/"data"/"processed"; CKPT=PROJ/"models"/"checkpoints"; CKPT.mkdir(parents=True,exist_ok=True)
FIG=PROJ/"results"/"figures"; FIG.mkdir(parents=True,exist_ok=True)
print("device:",DEVICE,"| GPU:",torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# --- 매핑 + 데이터 + 원인 라벨 ---
mp=yaml.safe_load(open(PROJ/"configs"/"mappings"/"defect_to_cause.yaml",encoding="utf-8"))
CAUSE_NAMES=list(mp["causes"])+["normal"]; cause2idx={c:i for i,c in enumerate(CAUSE_NAMES)}
NUM_CAUSE=len(CAUSE_NAMES)
def d2c(name): return cause2idx["normal"] if name=="none" else cause2idx[mp["pattern_to_cause"][name]["primary"]]
cause_of=np.array([d2c(CLASSES[d]) for d in range(NUM)],dtype=np.int64)
ACTION=mp["action_guide"]

def load_split(n):
    d=np.load(PROC/f"wafer_{n}.npz",allow_pickle=True); return d["X"],d["y"]
Xtr,ytr=load_split("train"); Xva,yva=load_split("val"); Xte,yte=load_split("test")
ctr,cva,cte=cause_of[ytr],cause_of[yva],cause_of[yte]
counts=np.bincount(ytr,minlength=NUM)
print("train:",Xtr.shape,"| 결함",NUM,"| 원인",NUM_CAUSE)

EPOCHS=50; PATIENCE=8; BATCH=256; LR=3e-4; ALPHA=0.5; FOCAL_GAMMA=2.0; CB_BETA=0.999
torch.manual_seed(42); np.random.seed(42)

class WaferDS(Dataset):
    def __init__(s,X,yd,yc,train=False): s.X=X;s.yd=yd;s.yc=yc;s.train=train
    def __len__(s): return len(s.X)
    def __getitem__(s,i):
        img=s.X[i].astype(np.float32)/2.0
        if s.train:
            k=np.random.randint(0,4)
            if k: img=np.rot90(img,k).copy()
            if np.random.rand()<0.5: img=np.fliplr(img).copy()
            if np.random.rand()<0.5: img=np.flipud(img).copy()
        img=(img-0.5)/0.5; img=np.expand_dims(img,0).repeat(3,0)
        return torch.from_numpy(img), int(s.yd[i]), int(s.yc[i])
mk=lambda X,yd,yc,tr: DataLoader(WaferDS(X,yd,yc,tr),batch_size=BATCH,shuffle=tr,num_workers=0,pin_memory=(DEVICE=="cuda"))
tr_loader=mk(Xtr,ytr,ctr,True); va_loader=mk(Xva,yva,cva,False)

In [ ]:
# --- 멀티태스크 모델 (백본 공유 + 2헤드) ---
class MultiTaskNet(nn.Module):
    def __init__(s,nd,nc):
        super().__init__()
        try: bb=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        except Exception: bb=models.resnet18(pretrained=True)
        inf=bb.fc.in_features; bb.fc=nn.Identity(); s.backbone=bb
        s.defect_head=nn.Linear(inf,nd); s.cause_head=nn.Linear(inf,nc)
    def forward(s,x):
        f=s.backbone(x); return s.defect_head(f), s.cause_head(f)
net=MultiTaskNet(NUM,NUM_CAUSE).to(DEVICE)

eff=1-np.power(CB_BETA,counts); cbw=(1-CB_BETA)/np.maximum(eff,1e-12)
cbw=(cbw/cbw.sum()*NUM).astype(np.float32); defect_w=torch.tensor(cbw,device=DEVICE)
class FocalLoss(nn.Module):
    def __init__(s,weight=None,gamma=2.0): super().__init__(); s.w=weight; s.g=gamma
    def forward(s,logits,target):
        logp=F.log_softmax(logits,1); ce=F.nll_loss(logp,target,weight=s.w,reduction="none")
        pt=logp.exp().gather(1,target.unsqueeze(1)).squeeze(1)
        return ((1-pt)**s.g*ce).mean()
defect_crit=FocalLoss(defect_w,FOCAL_GAMMA); cause_crit=nn.CrossEntropyLoss()
optimizer=torch.optim.AdamW(net.parameters(),lr=LR,weight_decay=1e-4)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)

In [ ]:
# --- 50에폭 학습 + 조기종료 (val 결함 F1 기준) ---
@torch.no_grad()
def quick_eval(loader):
    net.eval(); yd=[];yc=[];pd=[];pc=[]
    for xb,ydb,ycb in loader:
        od,oc=net(xb.to(DEVICE))
        pd.append(od.argmax(1).cpu().numpy()); pc.append(oc.argmax(1).cpu().numpy())
        yd.append(ydb.numpy()); yc.append(ycb.numpy())
    yd=np.concatenate(yd);yc=np.concatenate(yc);pd=np.concatenate(pd);pc=np.concatenate(pc)
    return f1_score(yd,pd,average="macro"), f1_score(yc,pc,average="macro")

best=-1; wait=0; best_path=CKPT/"q8_multitask_50ep.pt"; hist={"df1":[],"cf1":[]}
for ep in range(1,EPOCHS+1):
    net.train(); t0=time.time(); run=0.0
    for bi,(xb,ydb,ycb) in enumerate(tr_loader):
        xb,ydb,ycb=xb.to(DEVICE),ydb.to(DEVICE),ycb.to(DEVICE)
        od,oc=net(xb); loss=defect_crit(od,ydb)+ALPHA*cause_crit(oc,ycb)
        optimizer.zero_grad(); loss.backward(); optimizer.step(); run+=loss.item()
        if bi%100==0: print(f"  ep{ep} {bi}/{len(tr_loader)} loss={loss.item():.3f}",end="\r")
    scheduler.step()
    df1,cf1=quick_eval(va_loader); hist["df1"].append(df1); hist["cf1"].append(cf1)
    flag=""
    if df1>best: best=df1; wait=0; torch.save({"model":net.state_dict(),"classes":CLASSES,"causes":CAUSE_NAMES},best_path); flag="  <- best"
    else: wait+=1
    print(f"[ep{ep}/{EPOCHS}] val 결함F1={df1:.3f} 원인F1={cf1:.3f} | {time.time()-t0:.0f}s{flag}")
    if wait>=PATIENCE: print(f"\n조기종료 (best val 결함F1={best:.4f})"); break
print("\nbest val 결함 macro-F1:",round(best,4))

In [ ]:
# --- TTA 추론 (두 헤드 모두) + Q5 대비 비교 ---
net.load_state_dict(torch.load(best_path,map_location=DEVICE)["model"]); net.eval()

@torch.no_grad()
def predict_both(X, tta=True):
    transforms=[(k,f) for k in range(4) for f in [False,True]] if tta else [(0,False)]
    pd=np.zeros((len(X),NUM),dtype=np.float32); pc=np.zeros((len(X),NUM_CAUSE),dtype=np.float32)
    for i in range(0,len(X),512):
        base=X[i:i+512].astype(np.float32)/2.0; ad=None; ac=None
        for k,f in transforms:
            arr=base
            if k: arr=np.rot90(arr,k,axes=(1,2))
            if f: arr=arr[:,:,::-1]
            arr=(np.ascontiguousarray(arr)-0.5)/0.5; arr=np.expand_dims(arr,1).repeat(3,1)
            od,oc=net(torch.from_numpy(arr).to(DEVICE))
            od=od.softmax(1).cpu().numpy(); oc=oc.softmax(1).cpu().numpy()
            ad=od if ad is None else ad+od; ac=oc if ac is None else ac+oc
        pd[i:i+512]=ad/len(transforms); pc[i:i+512]=ac/len(transforms)
    return pd.argmax(1), pc.argmax(1)

pd_t,pc_t=predict_both(Xte,tta=True)
df1=f1_score(yte,pd_t,average="macro"); cf1=f1_score(cte,pc_t,average="macro")
dacc=accuracy_score(yte,pd_t); cacc=accuracy_score(cte,pc_t)
print("="*50)
print(f"  Q5 (6에폭)        결함F1=0.890  원인F1=0.896")
print(f"  Q8 (50ep+TTA)     결함F1={df1:.3f}  원인F1={cf1:.3f}")
print(f"  결함 acc={dacc:.3f} | 원인 acc={cacc:.3f}")
print("="*50)
print(f"  원인 F1 향상: {cf1-0.896:+.3f}")

In [ ]:
# --- 공정 원인 분류 리포트 + 혼동행렬 ---
present=sorted(set(cte.tolist())|set(pc_t.tolist()))
print("=== 공정 원인 (TTA) ===")
print(classification_report(cte,pc_t,labels=present,target_names=[CAUSE_NAMES[i] for i in present],digits=3,zero_division=0))
cm=confusion_matrix(cte,pc_t,labels=present); cmn=cm/cm.sum(1,keepdims=True).clip(min=1)
names=[CAUSE_NAMES[i] for i in present]
fig,ax=plt.subplots(figsize=(8,7)); im=ax.imshow(cmn,cmap="Blues",vmin=0,vmax=1)
ax.set_xticks(range(len(present))); ax.set_xticklabels(names,rotation=45,ha="right")
ax.set_yticks(range(len(present))); ax.set_yticklabels(names)
ax.set_xlabel("예측"); ax.set_ylabel("실제"); ax.set_title(f"Q8 원인 혼동행렬 (macro-F1={f1_score(cte,pc_t,average='macro'):.3f})")
for i,j in itertools.product(range(len(present)),range(len(present))):
    ax.text(j,i,f"{cmn[i,j]:.2f}",ha="center",va="center",color="white" if cmn[i,j]>0.5 else "black",fontsize=7)
fig.colorbar(im,fraction=0.046); plt.tight_layout(); plt.savefig(FIG/"q8_cause_confusion.png",dpi=110,bbox_inches="tight"); plt.show()

## 정리

- 멀티태스크 모델에 장기학습 + TTA를 적용해 결함·원인 macro-F1을 함께 끌어올렸다.
- 원인 F1은 결함 F1과 거의 같이 움직인다(1:1 매핑이라 독립적 추가 신호 없음). 결함 천장(~0.914)이 곧 원인 천장이다.
- 결과가 만족스러우면 이 체크포인트(`q8_multitask_50ep.pt`)를 최종 멀티태스크 모델로 삼아 `project`/README/REPORT의 원인 수치를 갱신하고, Streamlit 앱이 이 체크포인트를 쓰도록 바꾸면 된다.
- 앱 연동 시: `app/streamlit_app.py`의 `CKPT` 경로를 `q8_multitask_50ep.pt`로 변경.